In [66]:
import pandas as pd

df = pd.read_csv('new_combined.csv')

In [67]:
df.head()

,id,time,amplitude1_1,amplitude2_1,amplitude3_1,amplitude4_1,amplitude5_1,amplitude6_1,amplitude7_1,amplitude8_1,...,phase58_2,phase59_2,phase60_2,phase61_2,phase62_2,phase63_2,phase64_2,rssi_1,rssi_2,activity
0,368,10:16:40,118.726577,5.0,28.635642,30.805844,32.015621,32.893768,31.780497,30.886890,...,1.467715,1.532354,1.527345,1.515298,1.418147,1.212026,0.643501,-51,-48,4
1,369,10:16:40,118.726577,5.0,29.000000,31.144823,32.649655,32.202484,31.906112,30.413813,...,-2.944197,-2.922924,-2.886410,-2.934096,-2.999696,3.017238,2.356194,-51,-48,4
2,370,10:16:40,118.726577,5.0,29.154759,31.384710,32.449961,33.421550,32.557641,31.144823,...,-1.945130,-1.904041,-1.905533,-1.941688,-2.003204,-2.356194,-2.999696,-51,-48,4
3,371,10:16:40,118.726577,5.0,22.090722,24.020824,25.000000,25.079872,24.331050,23.537205,...,2.572279,2.594804,2.601173,2.582993,2.512796,2.191046,1.570796,-51,-48,4
4,372,10:16:40,118.726577,5.0,29.832868,31.780497,32.557641,33.136083,32.015621,31.064449,...,-2.910502,-2.889038,-2.853537,-2.875341,-2.966920,3.058451,2.622447,-51,-48,4


In [68]:
forPredict = []

In [69]:
import joblib

save_dir = './models/'

gmm_models = []
for i in range(6):
    model_filename = f'gmm_model_{i}.pkl'
    model_path = save_dir + model_filename
    loaded_gmm_model = joblib.load(model_path)
    gmm_models.append(loaded_gmm_model)

In [70]:
import joblib

pca = joblib.load('./models/pca_model.joblib')

In [71]:
import pywt
import numpy as np

wavelet_name = 'gaus1'

scales = np.arange(1, 128)	

In [72]:
Y = []

In [73]:
import numpy as np

def getWavelet(data):
	X = data.drop(columns = ['activity', 'id', 'time'])
	Y.append(data['activity'].iloc[0])
	pca_X = pd.DataFrame(pca.transform(X))
	coefficients, frequencies = pywt.cwt(pca_X.iloc[: , 0], scales, wavelet_name)
	wavelet_variances = np.sum(coefficients**2, axis=0)
	return wavelet_variances



In [74]:
def get25(group):
	fraction = len(group) /25
	forPredict = []
	for i in range(1,26):
		index = int(i * fraction)
		forPredict.append(group.iloc[index-1])

	return forPredict

In [75]:
def getPredictions(data):
	predicted_probs = []
	for i in range(6):
		predicted_probs.append(gmm_models[i].score_samples(data))
	predicted_labels = np.argmax(predicted_probs)
	return predicted_labels

In [76]:
recent = []

def addRecent(data):
	if len(recent) == 5:
		recent.pop(0)
	recent.append(data)



In [77]:
def predictionMapping(prediction):
	if prediction == 0 or prediction == 4:
		return 0
	elif prediction == 5:
		return 1
	else:
		return 2


In [78]:
def predict(prediction):
    prediction = predictionMapping(prediction)
    if len(recent) > 2:

        if recent[len(recent)-1] == recent[len(recent)-2] == prediction:
            addRecent(prediction)
            return prediction
        elif recent[len(recent)-2] == recent[len(recent)-3] == prediction:
            addRecent(prediction)
            return prediction
        else:
            if len(recent) > 3:
                if recent[len(recent)-1] == recent[len(recent)-3] == prediction:
                    addRecent(prediction)
                    return prediction
                else:
                    addRecent(prediction)
                    return recent[len(recent)-2]
            else:
                addRecent(prediction)
                return recent[len(recent)-2]
    else:
        addRecent(prediction)
        return prediction

In [58]:
# testData = [0,0,0,1,1,2,2,0,2,2,4,4,4,5,5,0,5,4,3,3,3,0,3,4]

# pred = []

# for i in testData:
#     pred.append(predict(i))
    
# print(pred)

[0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 1, 1, 1, 1, 0, 2, 2, 2, 2, 2]


In [79]:
grouped_data = df.groupby('time')
i = 0
predictions = np.array([])
for time, group in grouped_data:
	if len(group) > 25:
		forPredict = get25(group)
		wavelet = getWavelet(pd.DataFrame(forPredict))
		wavelet = pd.DataFrame(wavelet.reshape(1, -1))
		wavelet.columns = [str(i) for i in range(0, 25)]
		pred = int(getPredictions(wavelet))
		predictions = np.append(predictions, predict(pred))
		


In [80]:
len(predictions)

4484

In [81]:
len(Y)

4484

In [82]:
recent= []

for i in range(len(Y)):
    Y[i] = predict(Y[i])
    

In [83]:
Y

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [84]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(Y, predictions)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.88


In [85]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Assuming you have true labels (y_actual) and predicted labels (y_pred)
# For each class (0, 1, 2, 3), calculate precision, recall, and F1 score
precision = precision_score(Y, predictions, average=None)
recall = recall_score(Y, predictions, average=None)
f1 = f1_score(Y, predictions, average=None)

# Print results for each class
for class_label in range(3):
    print(f"Class {class_label}:")
    print(f"Precision: {precision[class_label]:.4f}")
    print(f"Recall: {recall[class_label]:.4f}")
    print(f"F1 Score: {f1[class_label]:.4f}\n")


Class 0:
Precision: 0.8809
Recall: 0.9695
F1 Score: 0.9231

Class 1:
Precision: 0.7185
Recall: 0.6165
F1 Score: 0.6636

Class 2:
Precision: 0.9225
Recall: 0.8277
F1 Score: 0.8725

